# Using RxInfer with StaticArrays

[StaticArrays.jl](https://github.com/JuliaArrays/StaticArrays.jl) provides arrays whose size is part of their type: an `SVector{2, Float64}` or an `SMatrix{2, 2, Float64}` lives on the stack, and operations on it are unrolled at compile time. RxInfer's message update rules are written generically, so if the data, the priors and the constants of a model are static arrays, the messages and the posteriors computed from them are static arrays too.

This example shows that this works end to end on a linear Gaussian state-space model: we run the same inference with ordinary arrays and with static ones, check that every posterior stays static, and compare the results. It shows what is possible, not a speed-up: at the sizes where static arrays shine, most of the time of `infer` goes to the reactive engine itself, not to the arithmetic.

Static matrices need FastCholesky 1.4.5 or later, whose Cholesky factorisation of a static matrix tolerates the rounding that makes a product such as `A * Σ * A'` not exactly symmetric.

In [ ]:
using RxInfer, StaticArrays, LinearAlgebra, Random, StableRNGs, BenchmarkTools, Plots

## The model

A hidden state rotates by a fixed angle at every step and is observed with noise, as in the *Kalman filtering and smoothing* example:

$$\begin{aligned}
x_t &\sim \mathcal{N}(A x_{t-1}, P), \\
y_t &\sim \mathcal{N}(B x_t, Q).
\end{aligned}$$

The model takes its prior `x0` and the matrices `A`, `B`, `P` and `Q` as arguments, so the same model runs with either kind of array.

In [ ]:
@model function rotate_ssm(y, x0, A, B, P, Q)
    x_prior ~ x0
    x_prev = x_prior
    for i in 1:length(y)
        x[i] ~ MvNormalMeanCovariance(A * x_prev, P)
        y[i] ~ MvNormalMeanCovariance(B * x[i], Q)
        x_prev = x[i]
    end
end

We generate 300 observations of a two-dimensional state.

In [ ]:
function generate_data(rng, A, B, P, Q, n)
    x, y = Vector{Vector{Float64}}(undef, n), Vector{Vector{Float64}}(undef, n)
    previous = zeros(size(A, 1))
    for i in 1:n
        x[i] = rand(rng, MvNormal(A * previous, P))
        y[i] = rand(rng, MvNormal(B * x[i], Q))
        previous = x[i]
    end
    return x, y
end

θ = π / 35
A = [cos(θ) -sin(θ); sin(θ) cos(θ)]
B = diageye(2)
P = diageye(2)
Q = 25.0 * diageye(2)
x0 = MvNormalMeanCovariance(zeros(2), 100.0 * diageye(2))

x, y = generate_data(StableRNG(1234), A, B, P, Q, 300);

## Inference with ordinary arrays

In [ ]:
result = infer(model = rotate_ssm(x0 = x0, A = A, B = B, P = P, Q = Q), data = (y = y,))
typeof(last(result.posteriors[:x]))

## Inference with static arrays

The same inference with the observations as `SVector`s and the prior and the matrices as static arrays. Nothing else changes.

In [ ]:
static_y = [SVector{2}(yi) for yi in y]
static_A, static_B, static_P, static_Q = SMatrix{2, 2}(A), SMatrix{2, 2}(B), SMatrix{2, 2}(P), SMatrix{2, 2}(Q)
static_x0 = MvNormalMeanCovariance(SVector{2}(mean(x0)), SMatrix{2, 2}(cov(x0)))

static_result = infer(
    model = rotate_ssm(x0 = static_x0, A = static_A, B = static_B, P = static_P, Q = static_Q),
    data = (y = static_y,),
)
typeof(last(static_result.posteriors[:x]))

Every posterior keeps static arrays for its mean and its covariance, and they agree with the posteriors computed with ordinary arrays up to rounding:

In [ ]:
is_static(q) = mean(q) isa SVector && cov(q) isa SMatrix
println("every posterior is static: ", all(is_static, static_result.posteriors[:x]))

difference = maximum(zip(result.posteriors[:x], static_result.posteriors[:x])) do (q, static_q)
    return max(maximum(abs, mean(q) - mean(static_q)), maximum(abs, cov(q) - cov(static_q)))
end
println("largest difference from the ordinary arrays: ", difference)

In [ ]:
means = mean.(static_result.posteriors[:x])
stds = map(q -> sqrt.(diag(cov(q))), static_result.posteriors[:x])
plot(getindex.(x, 1), label = "hidden state", color = :black)
scatter!(getindex.(y, 1), label = "observations", markersize = 2, alpha = 0.4)
plot!(getindex.(means, 1), ribbon = getindex.(stds, 1), label = "posterior (static arrays)", xlabel = "time", ylabel = "first coordinate")

## A larger state

The same works for any fixed size. Here a four-dimensional state rotates by a random orthogonal matrix.

In [ ]:
rng = StableRNG(42)
A4 = Matrix(qr(randn(rng, 4, 4)).Q)
_, y4 = generate_data(rng, A4, diageye(4), diageye(4), 25.0 * diageye(4), 100)

result4 = infer(
    model = rotate_ssm(
        x0 = MvNormalMeanCovariance(@SVector(zeros(4)), SMatrix{4, 4}(100.0 * diageye(4))),
        A = SMatrix{4, 4}(A4),
        B = SMatrix{4, 4}(diageye(4)),
        P = SMatrix{4, 4}(diageye(4)),
        Q = SMatrix{4, 4}(25.0 * diageye(4)),
    ),
    data = (y = [SVector{4}(yi) for yi in y4],),
)
println("every posterior is static: ", all(is_static, result4.posteriors[:x]))
typeof(last(result4.posteriors[:x]))

## Is it faster?

Not by much, at this size. Each message update rule is several times faster with static arrays and allocates nothing, but `infer` also builds the factor graph and runs the reactive engine, which costs the same with either kind of array and dominates for small states:

In [ ]:
ordinary = @benchmark infer(model = rotate_ssm(x0 = $x0, A = $A, B = $B, P = $P, Q = $Q), data = (y = $y,))
static = @benchmark infer(
    model = rotate_ssm(x0 = $static_x0, A = $static_A, B = $static_B, P = $static_P, Q = $static_Q),
    data = (y = $static_y,),
)
for (name, trial) in (("ordinary arrays", ordinary), ("static arrays", static))
    estimate = median(trial)
    println(rpad(name, 16), BenchmarkTools.prettytime(estimate.time), ", ", estimate.allocs, " allocations")
end

Static arrays become worthwhile when the arithmetic is a larger share of the work, for example with states of a dozen or more dimensions, and when the posteriors are passed on to code that benefits from static types. Above roughly 20 dimensions, compiling the unrolled code costs more than it saves.